# QQQ Opening Range Bias — v2 (refactored engine + statistics)

Rewrite of `QQQ_bias.ipynb` on top of the parameterised engine in `src/backtest.py`.

**Fixes vs v1**
- one engine instead of three copy-pasted loops;
- the paper replication runs on **all complete QQQ sessions** (no longer conditioned on NQ data availability);
- incomplete sessions dropped as **whole days**, never as single bars;
- DST-safe NQ timestamp handling; commissions modelled ($0.0005/share/side).

**New analyses (roadmap items)**
- placebo test: QQQ's own 09:25 pre-market bar instead of NQ;
- per-trade t-stats and bootstrap CI on Sharpe;
- per-year PnL breakdown;
- PnL-vs-slippage sensitivity curve.

⚠️ Requires the two CSVs in `../data/` — see `data/README.md`.

In [ ]:
import sys, datetime as dt
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import backtest as bt

CAPITAL = 25_000.0

## 1 · Data

QQQ drives the replication universe. NQ (and the day-intersection with QQQ) is used **only** by the filtered scenario, so the replication is no longer distorted by NQ data holes.

In [ ]:
qqq = bt.load_qqq("../data/QQQ_5min_10years_UTC.csv")
nq = bt.load_nq("../data/nq-10y-1min.csv")

qqq_days = bt.complete_days(qqq)
nq_days = bt.complete_days(nq)
both_days = qqq_days & nq_days

qqq_full = bt.restrict_to_days(qqq, qqq_days)      # replication universe
qqq_both = bt.restrict_to_days(qqq, both_days)     # filtered-scenario universe
nq_both = bt.restrict_to_days(nq, both_days)

print(f"complete QQQ sessions: {len(qqq_days)}")
print(f"complete NQ sessions:  {len(nq_days)}")
print(f"intersection:          {len(both_days)}")

## 2 · Scenarios

| # | universe | costs | confirmation |
|---|---|---|---|
| replication | all QQQ days | commission only | — |
| slippage | all QQQ days | + $0.02 entry / $0.04 stop | — |
| NQ filter | QQQ ∩ NQ days | + slippage | NQ 09:25 bar direction |
| placebo | all QQQ days | + slippage | QQQ **own** 09:25 pre-market bar |

The placebo is the control experiment: if it matches the NQ filter, the "cross-asset" story is really just pre-open momentum confirmation.

In [ ]:
SLIP = dict(entry_slippage=0.02, stop_slippage=0.04)

nq_confirm = bt.bar_direction_by_day(nq_both, dt.time(9, 25))
qqq_confirm = bt.bar_direction_by_day(qqq_full, dt.time(9, 25))

trades = {
    "replication": bt.run_backtest(qqq_full),
    "slippage": bt.run_backtest(qqq_full, **SLIP),
    "nq_filter": bt.run_backtest(qqq_both, confirm_dir=nq_confirm, **SLIP),
    "placebo_qqq925": bt.run_backtest(qqq_full, confirm_dir=qqq_confirm, **SLIP),
}
pd.DataFrame({k: {"trades": len(v), "net_pnl": v["pnl"].sum().round(0),
                  "pnl_per_share": v["pnl_per_share"].mean().round(4)}
              for k, v in trades.items()}).T

## 3 · Risk-adjusted metrics vs buy & hold

In [ ]:
def buy_hold_equity(df, capital=CAPITAL):
    px = df.groupby("day")["close"].last()
    eq = capital * px / px.iloc[0]
    eq.index = pd.to_datetime(eq.index)
    return eq

days_full = qqq_full["day"].unique()
equities = {k: bt.equity_curve(v, days_full if k != "nq_filter" else qqq_both["day"].unique())
            for k, v in trades.items()}
equities["buy_hold"] = buy_hold_equity(qqq_full)

metrics = pd.DataFrame({k: bt.compute_metrics(eq) for k, eq in equities.items()}).T
metrics.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for k, eq in equities.items():
    ax.plot(eq.index, eq.values, label=k, lw=1.5)
ax.set_yscale("log"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Equity curves (log scale)"); ax.set_ylabel("account value ($)")
plt.show()

## 4 · Statistical significance

The question the v1 notebook never asked: **is the edge distinguishable from zero?**

In [ ]:
pd.DataFrame({k: bt.trade_tstat(v) for k, v in trades.items()}).T.round(3)

In [ ]:
pd.DataFrame({k: bt.bootstrap_sharpe_ci(eq) for k, eq in equities.items()}).T.round(3)

## 5 · Per-year breakdown

If the PnL concentrates in 2020–2022 (as published ORB results are known to), the edge is a volatility-regime artifact, not a structural one.

In [ ]:
for k in ("replication", "nq_filter"):
    print(f"— {k}")
    display(bt.yearly_breakdown(trades[k]).round(3))

## 6 · Slippage sensitivity

Replaces the single $0.02 point estimate with the full curve: at what execution cost does the edge cross zero?

In [ ]:
sens = bt.slippage_sensitivity(qqq_full, trading_days=days_full)
display(sens.round(4))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sens["entry_slippage"], sens["net_pnl"], marker="o")
ax.axhline(0, color="gray", lw=1)
ax.set_xlabel("entry slippage ($/share, stop = 2x)"); ax.set_ylabel("net PnL ($)")
ax.set_title("Edge vs execution cost"); ax.grid(alpha=0.3)
plt.show()

## 7 · Exit-reason anatomy

The 10R target is quasi-decorative: most exits are EoD, i.e. the strategy is intraday momentum continuation with a 1R stop.

In [ ]:
pd.DataFrame({k: v["reason"].value_counts(normalize=True).round(3) for k, v in trades.items()}).T